# Train GNN

Original ver2 model and training settings. Install the package first; configure `SEAICE_CONFIG` before starting the kernel. This notebook trains only. Use the package CLI for ocean-masked manuscript evaluation.


## Export-only execution after kernel restart

To create the manuscript regional-zoom field archives without retraining:

1. Run setup/import/data cells through `## 1. Training Or Checkpoint Loading`.
2. Keep `DAILY_FORCE_RETRAIN = False` and `DAILY_LOAD_IF_EXISTS = True`.
3. Run the checkpoint-loading cell. It should print `Loading normalized final-SIC GNN checkpoint`.
4. Run `## 2. Evaluation Metric Functions`.
5. Skip full evaluation and metric-figure cells unless you need to regenerate metrics.
6. Run `## Export selected forecast fields for manuscript regional zoom`.

The export cell only forwards the two selected 2025 initialization cases and writes `.npz` field archives.



In [ ]:
from seaice_diagnostics.config import CACHE, ERA5, TRAINING, raw_path
# ============================================================
# DAILY CONTROL ENVIRONMENT AND IMPORTS
# ============================================================

import os
import glob
import math
import random
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, Subset

from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader
from torch_geometric.nn import HeteroConv, SAGEConv, Linear

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# ============================================================
# DAILY RAW DATA LOADING AND CACHE HELPERS
# ============================================================

@dataclass
class DataPaths:
    era5_sample: str = str(ERA5)
    era5_glob: str = raw_path("era5_glob")
    piomas_sic_glob: str = raw_path("piomas_sic_glob")
    piomas_sit_glob: str = raw_path("piomas_sit_glob")


def get_grid_coordinates(era5_sample_path: str) -> np.ndarray:
    ds = xr.open_dataset(era5_sample_path)
    lats = ds.latitude.values
    lons = ds.longitude.values
    ds.close()
    lon_grid, lat_grid = np.meshgrid(lons, lats)
    return np.column_stack([lat_grid.reshape(-1), lon_grid.reshape(-1)])


def create_grid_to_grid_edges(num_lat: int, num_lon: int) -> torch.Tensor:
    source_nodes = []
    target_nodes = []

    def node_id(i, j):
        return i * num_lon + j

    for i in tqdm(range(num_lat), desc="grid-grid edges"):
        for j in range(num_lon):
            src = node_id(i, j)
            for ni in [i - 1, i, i + 1]:
                if ni < 0 or ni >= num_lat:
                    continue
                for nj in [(j - 1) % num_lon, j, (j + 1) % num_lon]:
                    dst = node_id(ni, nj)
                    if dst != src:
                        source_nodes.append(src)
                        target_nodes.append(dst)
    return torch.tensor([source_nodes, target_nodes], dtype=torch.long)


def _cache_date_key(date_like) -> str:
    return pd.Timestamp(date_like).strftime("%Y%m%d")


def create_or_load_grid_daily_cache(
    era5_file_list,
    piomas_sic_file_list,
    piomas_sit_file_list,
    valid_dates,
    cache_dir,
    variables=None,
    overwrite=False,
):
    variables = variables or ["t2m", "d2m", "u10", "v10", "msl", "sst", "sic", "sit"]
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    expected_paths = [cache_dir / f"{_cache_date_key(d)}.npz" for d in valid_dates]
    if all(p.exists() for p in expected_paths) and not overwrite:
        print(f"Daily grid cache already complete: {cache_dir}")
        return cache_dir

    print("Preparing daily grid cache without scale separation:", cache_dir)
    ds_era5 = xr.open_mfdataset(era5_file_list, combine="by_coords", engine="netcdf4", parallel=False)
    ds_sic = xr.open_mfdataset(piomas_sic_file_list, combine="by_coords", engine="netcdf4", parallel=False).reindex(time=ds_era5.time, method="nearest")
    ds_sit = xr.open_mfdataset(piomas_sit_file_list, combine="by_coords", engine="netcdf4", parallel=False).reindex(time=ds_era5.time, method="nearest")
    ds_piomas = xr.merge([ds_sic, ds_sit], compat="override")

    era5_dates = pd.DatetimeIndex(ds_era5.time.values).normalize()
    date_to_idx = {pd.Timestamp(d).normalize(): i for i, d in enumerate(era5_dates)}
    missing_dates = []

    for date_like in tqdm(valid_dates, desc="cache daily fields"):
        date_norm = pd.Timestamp(date_like).normalize()
        cache_path = cache_dir / f"{_cache_date_key(date_norm)}.npz"
        if cache_path.exists() and not overwrite:
            continue
        if date_norm not in date_to_idx:
            missing_dates.append(str(date_norm.date()))
            continue

        t_idx = date_to_idx[date_norm]
        daily_list = []
        sic_full = None
        for var in variables:
            if var in ["sic", "sit"]:
                arr_2d = ds_piomas[var].isel(time=t_idx).values.astype(np.float32)
            else:
                arr_2d = ds_era5[var].isel(time=t_idx).values.astype(np.float32)
            daily_list.append(arr_2d)
            if var == "sic":
                sic_full = arr_2d.copy()

        np.savez_compressed(
            cache_path,
            daily=np.stack(daily_list, axis=0).astype(np.float32),
            sic_full=sic_full.astype(np.float32),
        )

    ds_era5.close()
    ds_sic.close()
    ds_sit.close()
    ds_piomas.close()

    if missing_dates:
        raise ValueError(f"Missing dates while generating daily grid cache. Examples: {missing_dates[:5]}")
    print("Daily grid cache ready:", cache_dir)
    return cache_dir



In [ ]:
# ============================================================
# RESEARCH CONFIGURATION, DATA SETUP, AND EVALUATION DOMAINS
# ============================================================

DAILY_HISTORY_LEN = 15
DAILY_HORIZON_DAYS = 30
DAILY_HIDDEN = 64
DAILY_NUM_LAYERS = 3
DAILY_BATCH_SIZE = 1

# Fair-comparison GNN update:
# - keep the old experiment outputs untouched by writing to a new tag.
# - train automatically on the first run because the new loss/normalization
#   makes old GNN checkpoints incompatible as a fair comparison.
DAILY_FULL_EPOCHS = 20
DAILY_PATIENCE = 3
DAILY_FORCE_RETRAIN = False
DAILY_LOAD_IF_EXISTS = True
DAILY_OVERWRITE_GRID_CACHE = False

EXPERIMENT_START_DATE = "1979-01-01"
EXPERIMENT_END_DATE = "2025-12-31"
TRAIN_START_DATE = "1979-01-01"
TRAIN_END_DATE = "2019-12-31"
VAL_START_DATE = "2020-01-01"
VAL_END_DATE = "2022-12-31"
TEST_START_DATE = "2023-01-01"
TEST_END_DATE = "2025-12-31"
TEST_YEARS = [2023, 2024, 2025]
FIGURE_YEARS = [2025]
FOCUSED_FIGURE_YEAR = 2025

# Unweighted control loss configuration.
# All valid ocean cells receive equal loss contribution. MIZ is used only in evaluation.
MODEL_TAG = "gnn_gridonly_epoch20_patience3_1979_2025_ver2"
ICE_EDGE_THRESHOLD = 0.15

# Same input-channel normalization idea as the CNN/U-Net baselines.
# Statistics are computed only from the training period.
CHANNEL_STATS_SAMPLE_STEP = 7
FORCE_REBUILD_CHANNEL_STATS = False

EXPERIMENT_NAME = "seaice_prediction"
DAILY_OUTPUT_DIR = (TRAINING / "gnn")
DAILY_FIGURE_DIR = (TRAINING / "gnn" / "figures")
DAILY_CHECKPOINT_DIR = DAILY_OUTPUT_DIR / "checkpoints"
for _path in [DAILY_OUTPUT_DIR, DAILY_FIGURE_DIR, DAILY_CHECKPOINT_DIR]:
    _path.mkdir(parents=True, exist_ok=True)

channel_stats_path = DAILY_OUTPUT_DIR / "channel_statistics.npz"

for _year in TEST_YEARS:
    (DAILY_OUTPUT_DIR / f"test_{_year}").mkdir(parents=True, exist_ok=True)
    (DAILY_FIGURE_DIR / f"test_{_year}").mkdir(parents=True, exist_ok=True)

paths = DataPaths()
era5_file_list = sorted(glob.glob(paths.era5_glob))
piomas_sic_files = sorted(glob.glob(paths.piomas_sic_glob))
piomas_sit_files = sorted(glob.glob(paths.piomas_sit_glob))

print("ERA5 file count:", len(era5_file_list))
print("PIOMAS SIC file count:", len(piomas_sic_files))
print("PIOMAS SIT file count:", len(piomas_sit_files))
if not era5_file_list or not piomas_sic_files or not piomas_sit_files:
    raise FileNotFoundError("Missing ERA5 or PIOMAS source files. Check DataPaths globs.")

# Catch interrupted or placeholder NetCDF files before xarray fails with a less clear error.
MIN_NETCDF_BYTES = 1_000_000
suspect_era5_files = [p for p in era5_file_list if Path(p).stat().st_size < MIN_NETCDF_BYTES]
suspect_piomas_files = [
    p for p in [*piomas_sic_files, *piomas_sit_files]
    if Path(p).stat().st_size < MIN_NETCDF_BYTES
]
if suspect_era5_files or suspect_piomas_files:
    message = "Some 1979-2025 input files look incomplete. Re-download or regenerate them before running this notebook.\n"
    message += "ERA5: " + ", ".join(map(str, suspect_era5_files[:10])) + "\n"
    message += "PIOMAS: " + ", ".join(map(str, suspect_piomas_files[:10]))
    raise ValueError(message)

all_era5 = xr.open_mfdataset(era5_file_list, combine="by_coords", engine="netcdf4", parallel=False)
all_dates = pd.DatetimeIndex(all_era5.time.values).normalize()
date_mask = (
    (all_dates >= pd.Timestamp(EXPERIMENT_START_DATE))
    & (all_dates <= pd.Timestamp(EXPERIMENT_END_DATE))
)
valid_dates = pd.DatetimeIndex(all_dates[date_mask])
lat_values = np.asarray(all_era5.latitude.values)
lon_values = np.asarray(all_era5.longitude.values)
num_lat = len(lat_values)
num_lon = len(lon_values)
all_era5.close()

expected_dates = pd.date_range(EXPERIMENT_START_DATE, EXPERIMENT_END_DATE, freq="D")
if not valid_dates.equals(expected_dates):
    missing = expected_dates.difference(valid_dates)
    raise ValueError(f"ERA5 daily coverage is incomplete for {EXPERIMENT_START_DATE} to {EXPERIMENT_END_DATE}. Missing examples: {missing[:5].tolist()}")
print("valid_dates:", len(valid_dates), valid_dates.min(), "~", valid_dates.max())


grid_coords = get_grid_coordinates(paths.era5_sample)
edge_grid2grid = create_grid_to_grid_edges(num_lat=num_lat, num_lon=num_lon)
edge_indices = (edge_grid2grid,)

used_variables = ["t2m", "d2m", "u10", "v10", "msl", "sst", "sic", "sit"]
grid_daily_cache_dir = str(CACHE)
create_or_load_grid_daily_cache(
    era5_file_list=era5_file_list,
    piomas_sic_file_list=piomas_sic_files,
    piomas_sit_file_list=piomas_sit_files,
    valid_dates=valid_dates,
    cache_dir=grid_daily_cache_dir,
    variables=used_variables,
    overwrite=DAILY_OVERWRITE_GRID_CACHE,
)


def build_or_load_channel_stats(cache_dir, valid_dates, train_start, train_end, sample_step=7, force=False):
    if channel_stats_path.exists() and not force:
        stats = np.load(channel_stats_path)
        print("Loading GNN channel stats:", channel_stats_path)
        return stats["mean"].astype(np.float32), stats["std"].astype(np.float32)

    dates = pd.date_range(train_start, train_end, freq="D")[::max(1, int(sample_step))]
    sums = np.zeros(len(used_variables), dtype=np.float64)
    sums_sq = np.zeros(len(used_variables), dtype=np.float64)
    counts = np.zeros(len(used_variables), dtype=np.float64)

    for date in tqdm(dates, desc="GNN training-period channel stats"):
        with np.load(Path(cache_dir) / f"{_cache_date_key(date)}.npz") as npz:
            arr = npz["daily"].astype(np.float64)
        for v in range(len(used_variables)):
            vals = arr[v]
            valid = np.isfinite(vals)
            if valid.any():
                x = vals[valid]
                sums[v] += x.sum()
                sums_sq[v] += np.square(x).sum()
                counts[v] += valid.sum()

    mean = sums / np.maximum(counts, 1)
    var = sums_sq / np.maximum(counts, 1) - np.square(mean)
    std = np.sqrt(np.maximum(var, 1e-12))
    std = np.where(std < 1e-6, 1.0, std)

    np.savez(
        channel_stats_path,
        variables=np.asarray(used_variables),
        mean=mean.astype(np.float32),
        std=std.astype(np.float32),
    )
    print("Saved GNN channel stats:", channel_stats_path)
    print(pd.DataFrame({"variable": used_variables, "mean": mean, "std": std}))
    return mean.astype(np.float32), std.astype(np.float32)


channel_mean, channel_std = build_or_load_channel_stats(
    grid_daily_cache_dir,
    valid_dates,
    TRAIN_START_DATE,
    TRAIN_END_DATE,
    sample_step=CHANNEL_STATS_SAMPLE_STEP,
    force=FORCE_REBUILD_CHANNEL_STATS,
)

# These boxes support regional diagnosis of the same Pan-Arctic prediction.
# They are not separate decoder heads and are not a vessel route geometry.
NSR_REGION_BOXES = [
    # name, lat_min, lat_max, lon_min_0_360, lon_max_0_360
    ("chukchi", 66.0, 78.0, 180.0, 205.0),
    ("east_siberian", 68.0, 82.5, 140.0, 180.0),
    ("laptev", 70.0, 82.5, 100.0, 140.0),
    ("kara", 68.0, 82.5, 60.0, 100.0),
    ("barents", 68.0, 82.5, 20.0, 60.0),
]
NSR_REGION_NAMES = [box[0] for box in NSR_REGION_BOXES]


def _lon_to_360(lon):
    return np.mod(np.asarray(lon, dtype=np.float32), 360.0)


def _lon_between_360(lon360, lon_min, lon_max):
    lon_min = float(lon_min) % 360.0
    lon_max = float(lon_max) % 360.0
    if lon_min <= lon_max:
        return (lon360 >= lon_min) & (lon360 <= lon_max)
    return (lon360 >= lon_min) | (lon360 <= lon_max)


def build_nsr_region_masks(grid_coords, num_lat, num_lon, region_boxes):
    lat = np.asarray(grid_coords[:, 0], dtype=np.float32)
    lon360 = _lon_to_360(grid_coords[:, 1])
    masks = {}
    for name, lat_min, lat_max, lon_min, lon_max in region_boxes:
        in_region = (
            (lat >= float(lat_min))
            & (lat <= float(lat_max))
            & _lon_between_360(lon360, lon_min, lon_max)
        )
        masks[name] = in_region.reshape(num_lat, num_lon)
        print(f"NSR region {name:14s}: {int(in_region.sum()):6d} / {int(lat.shape[0])}")
    masks["nsr_corridor"] = np.logical_or.reduce([masks[name] for name in NSR_REGION_NAMES])
    print("NSR corridor union :", int(masks["nsr_corridor"].sum()), "/", int(lat.shape[0]))
    return masks


nsr_region_masks_2d = build_nsr_region_masks(grid_coords, num_lat, num_lon, NSR_REGION_BOXES)
analysis_domain_masks_2d = {
    "pan_arctic": np.ones((num_lat, num_lon), dtype=bool),
    "nsr_corridor": nsr_region_masks_2d["nsr_corridor"],
}
analysis_domain_masks_2d.update({name: nsr_region_masks_2d[name] for name in NSR_REGION_NAMES})
ANALYSIS_DOMAINS = list(analysis_domain_masks_2d.keys())
print("analysis domains:", ANALYSIS_DOMAINS)



In [ ]:
# ============================================================
# UNWEIGHTED FULL-ARCTIC DIRECT FORECAST DATASET
# ============================================================

class DailyGridOnlyGNNForecastDataset(Dataset):
    # Encoder inputs and forecast targets both retain the complete Arctic grid.
    def __init__(
        self,
        cache_dir,
        edge_indices,
        valid_dates,
        history_len=15,
        horizon=30,
        variables=None,
        channel_mean=None,
        channel_std=None,
        recent_cache_size=64,
    ):
        super().__init__()
        self.cache_dir = Path(cache_dir)
        self.valid_dates = pd.DatetimeIndex(valid_dates)
        self.history_len = int(history_len)
        self.horizon = int(horizon)
        (self.edge_grid2grid,) = edge_indices
        self.variables = variables or ["t2m", "d2m", "u10", "v10", "msl", "sst", "sic", "sit"]
        self.recent_cache_size = int(recent_cache_size)
        self._recent = {}

        if self.history_len < 2 or self.horizon < 1:
            raise ValueError("history_len must be >= 2 and horizon must be >= 1.")
        sample_path = self.cache_dir / f"{_cache_date_key(self.valid_dates[0])}.npz"
        if not sample_path.exists():
            raise FileNotFoundError(f"cache sample file not found: {sample_path}")
        sample_daily = np.load(sample_path)["daily"]
        self.num_lat = int(sample_daily.shape[1])
        self.num_lon = int(sample_daily.shape[2])
        self.num_grid_nodes = self.num_lat * self.num_lon
        self.num_vars = len(self.variables)
        self.channel_mean = None if channel_mean is None else np.asarray(channel_mean, dtype=np.float32)
        self.channel_std = None if channel_std is None else np.asarray(channel_std, dtype=np.float32)
        if self.channel_mean is not None and self.channel_mean.shape[0] != self.num_vars:
            raise ValueError("channel_mean length must match variables.")
        if self.channel_std is not None and self.channel_std.shape[0] != self.num_vars:
            raise ValueError("channel_std length must match variables.")
        self.grid_input_dim = self.history_len * self.num_vars
        print(
            "Pan-Arctic daily direct dataset ready:",
            f"history_len={self.history_len}",
            f"horizon={self.horizon}",
            f"len={len(self)}",
            f"grid_input_dim={self.grid_input_dim}",
        )

    def __len__(self):
        return len(self.valid_dates) - self.history_len - self.horizon + 1

    def _load_day(self, date_like):
        key = _cache_date_key(date_like)
        if key in self._recent:
            return self._recent[key]
        path = self.cache_dir / f"{key}.npz"
        if not path.exists():
            raise FileNotFoundError(f"cache file missing: {path}")
        arr = np.load(path)
        day = {"daily": arr["daily"], "sic_full": arr["sic_full"]}
        self._recent[key] = day
        if len(self._recent) > self.recent_cache_size:
            self._recent.pop(next(iter(self._recent)))
        return day

    def current_pos(self, idx):
        return int(idx) + self.history_len - 1

    def target_positions(self, idx):
        curr = self.current_pos(idx)
        return np.arange(curr + 1, curr + 1 + self.horizon, dtype=int)

    def current_date(self, idx):
        return pd.Timestamp(self.valid_dates[self.current_pos(idx)])

    def target_dates(self, idx):
        return pd.DatetimeIndex(self.valid_dates[self.target_positions(idx)])

    def __getitem__(self, idx):
        idx = int(idx)
        hist_positions = np.arange(idx, idx + self.history_len, dtype=int)
        target_positions = self.target_positions(idx)
        hist_days = [self._load_day(self.valid_dates[pos]) for pos in hist_positions]
        target_days = [self._load_day(self.valid_dates[pos]) for pos in target_positions]

        daily_hist = np.stack([day["daily"] for day in hist_days], axis=0)
        full_hist = np.stack([day["sic_full"] for day in hist_days], axis=0)
        full_target = np.stack([day["sic_full"] for day in target_days], axis=0)
        curr_full_sic = full_hist[-1]

        if self.channel_mean is not None and self.channel_std is not None:
            daily_hist = (daily_hist - self.channel_mean[None, :, None, None]) / self.channel_std[None, :, None, None]

        grid_x = daily_hist.reshape(self.history_len * self.num_vars, -1).T
        target_delta_flat = (full_target - curr_full_sic[None, :, :]).reshape(self.horizon, -1).T
        full_target_flat = full_target.reshape(self.horizon, -1).T
        curr_full_flat = curr_full_sic.reshape(-1, 1)

        data = HeteroData()
        data["grid"].x = torch.tensor(grid_x, dtype=torch.float32)
        ocean_mask = np.isfinite(full_target_flat)

        data["grid"].y_delta = torch.tensor(target_delta_flat, dtype=torch.float32)
        data["grid"].y_full = torch.tensor(full_target_flat, dtype=torch.float32)
        data["grid"].curr_full = torch.tensor(curr_full_flat, dtype=torch.float32)
        data["grid"].ocean_mask = torch.tensor(ocean_mask, dtype=torch.bool)
        data["grid"].num_nodes = data["grid"].x.size(0)
        data["grid", "flows_to", "grid"].edge_index = self.edge_grid2grid
        return data


def get_direct_target_date_frame(dataset):
    rows = []
    for idx in range(len(dataset)):
        targets = dataset.target_dates(idx)
        rows.append(
            {
                "sample_idx": idx,
                "init_date": dataset.current_date(idx).normalize(),
                "target_start": targets[0].normalize(),
                "target_end": targets[-1].normalize(),
            }
        )
    return pd.DataFrame(rows)


def split_direct_dataset_by_complete_target_window(dataset):
    frame = get_direct_target_date_frame(dataset)
    split_windows = {
        "train": (TRAIN_START_DATE, TRAIN_END_DATE),
        "val": (VAL_START_DATE, VAL_END_DATE),
        "test": (TEST_START_DATE, TEST_END_DATE),
    }
    result = {}
    print("===== complete-target-window train/val/test split =====")
    print(f"history_len={dataset.history_len}, horizon={dataset.horizon}")
    for name, (start, end) in split_windows.items():
        start = pd.Timestamp(start).normalize()
        end = pd.Timestamp(end).normalize()
        idx = frame.loc[
            (frame["target_start"] >= start) & (frame["target_end"] <= end),
            "sample_idx",
        ].astype(int).tolist()
        if not idx:
            raise ValueError(f"Empty {name} split detected.")
        result[name] = Subset(dataset, idx)
        sub = frame.loc[frame["sample_idx"].isin(idx)]
        print(
            f"{name:5s}: {len(idx):5d} samples | "
            f"target window {sub.target_start.min().date()} ~ {sub.target_end.max().date()}"
        )
    return result["train"], result["val"], result["test"], frame



In [ ]:
# ============================================================
# FULL-ARCTIC DETERMINISTIC GRID-ONLY GNN AND UNWEIGHTED FINAL-SIC TRAINING
# ============================================================

class DailyGridOnlyGNNEncoder(nn.Module):
    def __init__(self, in_grid, hidden=64, num_layers=2):
        super().__init__()
        self.grid_bn = nn.BatchNorm1d(in_grid)
        self.grid_encoder = Linear(in_grid, hidden)
        self.convs = nn.ModuleList([
            HeteroConv(
                {
                    ("grid", "flows_to", "grid"): SAGEConv(hidden, hidden),
                },
                aggr="mean",
            )
            for _ in range(num_layers)
        ])

    def forward(self, x_grid, edge_index_dict):
        x_grid = torch.nan_to_num(x_grid, nan=0.0, posinf=0.0, neginf=0.0)
        h = {
            "grid": self.grid_encoder(self.grid_bn(x_grid)).relu(),
        }
        for conv in self.convs:
            h = conv(h, edge_index_dict)
            h = {k: v.relu() for k, v in h.items()}
        return h["grid"]


class DailyGridOnlyGNNForecastModel(nn.Module):
    def __init__(self, in_grid, horizon=30, hidden=64, num_layers=2):
        super().__init__()
        self.encoder = DailyGridOnlyGNNEncoder(in_grid, hidden=hidden, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, horizon),
        )

    def forward(self, x_grid, edge_index_dict):
        h_grid = self.encoder(x_grid, edge_index_dict)
        return {"delta_full": self.head(h_grid)}


def masked_mse_loss_multi(pred, target, mask=None):
    valid = torch.isfinite(target)
    if mask is not None:
        valid = valid & mask.bool()
    if int(valid.sum().item()) == 0:
        return pred.sum() * 0.0
    return ((pred - target) ** 2)[valid].mean()


def compute_full_loss(outputs, batch):
    mask = batch["grid"].ocean_mask
    pred_full = torch.clamp(batch["grid"].curr_full + outputs["delta_full"], 0.0, 1.0)
    loss = masked_mse_loss_multi(pred_full, batch["grid"].y_full, mask)
    valid_count = int((mask & torch.isfinite(batch["grid"].y_full)).sum().item())
    return loss, {"loss": float(loss.detach().item()), "valid_ocean_cells": float(valid_count)}


def _forward_full_model(model, batch):
    return model(batch["grid"].x, batch.edge_index_dict)


def run_full_epoch(model, loader, device, optimizer=None, epoch_idx=None, epochs_total=None):
    is_train = optimizer is not None
    model.train(is_train)
    aggregate = {}
    count = 0
    phase = "train" if is_train else "val"
    pbar = tqdm(loader, desc=f"{phase} full {epoch_idx}/{epochs_total}", leave=False)
    for batch in pbar:
        batch = batch.to(device)
        if is_train:
            optimizer.zero_grad(set_to_none=True)
        outputs = _forward_full_model(model, batch)
        loss, metrics = compute_full_loss(outputs, batch)
        if is_train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        for key, value in metrics.items():
            aggregate[key] = aggregate.get(key, 0.0) + float(value)
        count += 1
        pbar.set_postfix({"loss": f"{aggregate['loss'] / max(count, 1):.4f}"})
    return {key: value / max(count, 1) for key, value in aggregate.items()}


def fit_full_arctic_model(model, train_loader, val_loader, device, save_path, epochs=20, patience=3, lr=1e-3):
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    best_val = float("inf")
    wait = 0
    history = []
    for epoch in range(1, int(epochs) + 1):
        train_stats = run_full_epoch(model, train_loader, device, optimizer, epoch, epochs)
        val_stats = run_full_epoch(model, val_loader, device, None, epoch, epochs)
        row = {"epoch": epoch}
        row.update({f"train_{key}": value for key, value in train_stats.items()})
        row.update({f"val_{key}": value for key, value in val_stats.items()})
        history.append(row)
        print(
            f"[full Arctic normalized final-SIC loss] epoch {epoch:03d} | "
            f"train_loss={train_stats['loss']:.6f} | val_loss={val_stats['loss']:.6f}"
        )
        if float(val_stats["loss"]) < best_val:
            best_val = float(val_stats["loss"])
            wait = 0
            torch.save(model.state_dict(), save_path)
            print("  saved checkpoint:", save_path)
        else:
            wait += 1
            if wait >= patience:
                print("  early stopping after", wait, "non-improving epochs")
                break
    if save_path.exists():
        model.load_state_dict(torch.load(save_path, map_location=device))
    return pd.DataFrame(history)




## 1. Training Or Checkpoint Loading

This cell creates the dataset splits and model object. With the default flags it loads the trained checkpoint and skips training. Set `DAILY_FORCE_RETRAIN=True` only when intentionally starting a new training run.



In [ ]:
# ============================================================
# BUILD DATASET AND TRAIN ONE FULL-ARCTIC DETERMINISTIC MODEL
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

dataset_daily = DailyGridOnlyGNNForecastDataset(
    cache_dir=grid_daily_cache_dir,
    edge_indices=edge_indices,
    valid_dates=valid_dates,
    history_len=DAILY_HISTORY_LEN,
    horizon=DAILY_HORIZON_DAYS,
    variables=used_variables,
    channel_mean=channel_mean,
    channel_std=channel_std,
    recent_cache_size=64,
)
train_ds_daily, val_ds_daily, test_ds_daily, target_frame_daily = split_direct_dataset_by_complete_target_window(dataset_daily)
train_loader_daily = DataLoader(train_ds_daily, batch_size=DAILY_BATCH_SIZE, shuffle=True, num_workers=0)
val_loader_daily = DataLoader(val_ds_daily, batch_size=DAILY_BATCH_SIZE, shuffle=False, num_workers=0)

model_full_daily = DailyGridOnlyGNNForecastModel(
    in_grid=dataset_daily.grid_input_dim,
    horizon=dataset_daily.horizon,
    hidden=DAILY_HIDDEN,
    num_layers=DAILY_NUM_LAYERS,
).to(device)

full_ckpt = DAILY_CHECKPOINT_DIR / "best.pth"
if DAILY_LOAD_IF_EXISTS and full_ckpt.exists() and not DAILY_FORCE_RETRAIN:
    print("Loading normalized final-SIC grid-only GNN checkpoint:", full_ckpt)
    model_full_daily.load_state_dict(torch.load(full_ckpt, map_location=device))
    history_full_daily = None
else:
    if DAILY_FORCE_RETRAIN:
        print("Training normalized final-SIC grid-only GNN model from scratch:", full_ckpt)
    else:
        print("Checkpoint not found or loading disabled; training normalized final-SIC grid-only GNN model:", full_ckpt)
    history_full_daily = fit_full_arctic_model(
        model_full_daily,
        train_loader_daily,
        val_loader_daily,
        device,
        save_path=full_ckpt,
        epochs=DAILY_FULL_EPOCHS,
        patience=DAILY_PATIENCE,
        lr=1e-3,
    )
    history_path = DAILY_OUTPUT_DIR / "training_history_gnn_gridonly_epoch20_patience3_1979_2025_ver2.csv"
    history_full_daily.to_csv(history_path, index=False)
    print("saved:", history_path)

